# 02 · Rank species by iNaturalist data availability

For each species list (output of notebook 01), this notebook:
1. matches each name to an iNaturalist taxon (exact name, or a synonym iNaturalist knows) and reuses `taxon_id` when the list already has one,
2. counts observations in the search area (a bbox or an iNaturalist `place_id`): all observations and research grade. Research grade requires a photo, so for plants that equals usable photos,
3. writes a ranked CSV, sorted from most to least research-grade observations.

**Colab disconnects are fine.** Name matches and counts are cached to `WORKDIR` on Drive after every species, so re-running a cell picks up where it stopped.

iNaturalist allows roughly 1 request per second. The full bbox list (~294 species) takes about 13 min the first time, and the desert SW list about 8 min after that.

> **In this repository.** Second step of the data pipeline. It reads the two hybrid lists from `03_build_species_lists.ipynb`, looks each name up on iNaturalist and counts its observations. The two ranked CSVs it writes are committed as `data/reference/ranked_desert_sw.csv` and `data/reference/ranked_full_bbox.csv`, the input of `src/invasive_clf/data/build_dataset.py`. Needs internet access (about 20 minutes at iNaturalist's rate limit).

## Setup

In [ ]:
# Where files live. On Colab this mounts your Google Drive so outputs (and the iNat cache)
# survive disconnects; share the folder with your group. Off Colab it just uses the current folder.
import os
WORKDIR = "/content/drive/MyDrive/invasive_plants"   # change to your shared folder

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
except ImportError:
    pass
print("Working in", os.getcwd())

In [ ]:
import json, sys, time
from pathlib import Path
import pandas as pd
import requests

## Configure the runs
Each run is one species list plus one search area. Use `{"place_id": ...}` or a bbox `{"swlat":..., "swlng":..., "nelat":..., "nelng":...}`.

In [ ]:
RUNS = [
    dict(name="full bbox",
         species_csv="species_full_bbox.csv",
         geo=dict(swlat=31.3, swlng=-120.0, nelat=42.0, nelng=-103.0),
         out="ranked_full_bbox.csv"),
    dict(name="desert southwest",
         species_csv="species_desert_sw.csv",
         geo=dict(place_id=118341),
         out="ranked_desert_sw.csv"),
]
NAME_COL = "inat_name"
TAXA_CACHE = Path("inat_taxa_cache.json")     # name -> taxon match
COUNT_CACHE = Path("inat_count_cache.json")   # (area, taxon, filter) -> count

## iNaturalist API helpers

In [ ]:
API = "https://api.inaturalist.org/v1"
PLANTAE_ID = 47126
session = requests.Session()
session.headers["User-Agent"] = "SW-invasive-plants-course-project (Colab)"


def get(path, **params):
    """GET with a 1 s pause between calls and backoff on rate limits / server errors."""
    for attempt in range(5):
        try:
            r = session.get(f"{API}/{path}", params=params, timeout=30)
        except requests.RequestException as e:
            wait = 5 * (attempt + 1)
            print(f"    network error ({e.__class__.__name__}); retrying in {wait}s"); time.sleep(wait); continue
        if r.status_code == 429 or r.status_code >= 500:
            wait = 10 * (attempt + 1)
            print(f"    HTTP {r.status_code}; retrying in {wait}s"); time.sleep(wait); continue
        r.raise_for_status()
        time.sleep(1.0)
        return r.json()
    raise RuntimeError(f"giving up on {path} {params}")


def norm(name):
    return " ".join(name.replace("×", " ").lower().split())


def _pack(t, how):
    return dict(taxon_id=t["id"], inat_accepted_name=t["name"], inat_rank=t["rank"],
                inat_common=t.get("preferred_common_name"), match=how)


def resolve(name):
    """Match a scientific name to an active iNaturalist plant taxon."""
    results = get("taxa", q=name.replace("×", "").strip(), taxon_id=PLANTAE_ID,
                  per_page=30, all_names="true")["results"]
    target = norm(name)
    for t in results:                                   # accepted name matches exactly
        if norm(t["name"]) == target and t.get("is_active", True):
            return _pack(t, "exact")
    for t in results:                                   # matched through a synonym / older name
        if norm(t.get("matched_term", "")) == target and t.get("is_active", True):
            return _pack(t, f"synonym of {t['name']}")
    return dict(taxon_id=None, inat_accepted_name=None, inat_rank=None, inat_common=None, match="NOT FOUND")


def load(p):
    return json.loads(p.read_text()) if p.exists() else {}

taxa_cache, count_cache = load(TAXA_CACHE), load(COUNT_CACHE)


def count(taxon_id, geo, **filters):
    key = json.dumps([geo, taxon_id, filters], sort_keys=True)
    if key not in count_cache:
        count_cache[key] = get("observations", taxon_id=taxon_id, per_page=0, **geo, **filters)["total_results"]
        COUNT_CACHE.write_text(json.dumps(count_cache))
    return count_cache[key]

## Check the search areas
Confirm the place name printed for `place_id` is the area you meant.

In [ ]:
for run in RUNS:
    if "place_id" in run["geo"]:
        p = get(f"places/{run['geo']['place_id']}")["results"]
        print(run["name"], "->", p[0].get("display_name") or p[0].get("name") if p else "NO SUCH PLACE")
    else:
        print(run["name"], "-> bbox", run["geo"])

## Ranking function

In [ ]:
def rank_species(species_csv, geo, out, name_col=NAME_COL):
    df = pd.read_csv(species_csv, keep_default_na=False)
    known = {}                                        # ids already in the list -> skip lookups
    if "taxon_id" in df.columns:
        for _, r in df.iterrows():
            if str(r["taxon_id"]).strip():
                known[r[name_col]] = dict(taxon_id=int(float(r["taxon_id"])),
                                          inat_accepted_name=r.get("inat_accepted_name") or None,
                                          inat_rank=None, inat_common=None, match="id from input")
        df = df.drop(columns=[c for c in ("taxon_id", "inat_accepted_name") if c in df.columns])

    rows, names = [], df[name_col].tolist()
    for i, name in enumerate(names, 1):
        if name in known:
            info = dict(taxa_cache.get(name) or known[name])
        else:
            if name not in taxa_cache:
                taxa_cache[name] = resolve(name)
                TAXA_CACHE.write_text(json.dumps(taxa_cache, indent=1))
            info = dict(taxa_cache[name])
        if info["taxon_id"]:
            info["obs_all"] = count(info["taxon_id"], geo)
            info["obs_research"] = count(info["taxon_id"], geo, quality_grade="research")
        else:
            info.update(obs_all=None, obs_research=None)
        info["obs_research_photos"] = info["obs_research"]   # research grade requires media
        rows.append(info)
        if i % 10 == 0 or i == len(names):
            print(f"  [{i:3d}/{len(names)}] {name:36s} {info['match']:30s} research = {info['obs_research']}")

    res = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    for c in ("taxon_id", "obs_all", "obs_research", "obs_research_photos"):
        res[c] = res[c].astype("Int64")
    dup = res["taxon_id"].notna() & res.duplicated("taxon_id", keep=False)
    res["duplicate_taxon"] = dup.map({True: "yes", False: ""})     # two names -> one iNat taxon
    res = res.sort_values(["obs_research", "obs_all"], ascending=False, na_position="last")
    res.insert(0, "rank_by_count", range(1, len(res) + 1))
    res.to_csv(out, index=False)
    return res


def summarize(res, name_col=NAME_COL):
    found = res[res["taxon_id"].notna()]
    label = "classifier_label" if "classifier_label" in res.columns else None
    rows = []
    for n in (5000, 2000, 1000, 500, 250, 100):
        hit = found[found["obs_research"] >= n]
        row = {"at least": n, "species": len(hit)}
        if label:
            row.update(hit[label].value_counts().to_dict())
        rows.append(row)
    display(pd.DataFrame(rows).fillna(0).set_index("at least").astype(int))
    issues = res[~res["match"].isin(["exact", "id from input"]) | (res["duplicate_taxon"] == "yes")]
    if len(issues):
        print("Check these (synonym match, not found, or two names -> one taxon):")
        display(issues[[name_col, "match", "inat_accepted_name", "taxon_id", "duplicate_taxon"]])

## Run

In [ ]:
results = {}
for run in RUNS:
    print(f"\n=== {run['name']} ===")
    results[run["name"]] = rank_species(run["species_csv"], run["geo"], run["out"])
    print("wrote", run["out"])

## Results

In [ ]:
show = [NAME_COL, "common_name", "classifier_label", "obs_research", "obs_all", "noxious_states"]
for name, res in results.items():
    print(f"\n=== {name}: species with at least N research-grade observations ===")
    summarize(res)
    display(res[["rank_by_count"] + [c for c in show if c in res.columns]].head(30))

Optional: download the ranked files. They're already saved in `WORKDIR` on Drive.

In [ ]:
try:
    from google.colab import files
    for run in RUNS:
        files.download(run["out"])
except ImportError:
    print("Not on Colab: files are in", os.getcwd())